# DCT Laboratory — Volume I, Chapter 2
## Enterprise Systems and Transformation
**Seed `26102`** · Companion to Chapter 2 and **AXIOM Lab 1.2** (module **AXIOM-02**) · Mirrored in `DCT_V1_Ch02_Lab.xlsx`

This laboratory reproduces **Worked Example 2.1, *Where Does Meridian End?***, with the book's declarations,
and works the five steps of AXIOM Lab 1.2: draw boundary A, move the dealers inside, change levels, close the
loop, map a loop. Every Meridian number below is the book's (Table 2.6, Figures 2.2–2.3, Online Appendix 2.B;
for the feedback bench, Meridian's efficiency in the Chapter 1 model, Tables 1.9 and 5.5); the final cell checks
them. Money is in \$ million.

## The declarations (Worked Example 2.1, Step 1)

Industrial Systems sells \$2.56 billion, 61 percent of Meridian's \$4.2 billion. Of this, \$1.54 billion, about
60 percent, is shipped to independent dealers, who resell to end customers at about 20 percent above Meridian's
price. The analysis needs six **entities** (suppliers, Meridian, the dealer network, end customers, capital
markets, the regulator) and nine **typed flows** $(e, e', \kappa)$, each of a type $\kappa$ in Definition 2.3's
list: capital, goods, services, information, personnel. The book values four of them a year (Table 2.6).

In [ ]:
import itertools
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SEED = 26102

REVENUE, IS_SHARE = 4200.0, 0.61          # Meridian's revenue; Industrial Systems' share (Chapter 1)
SHIP, MARKUP = 1540.0, 0.20                # shipped to independent dealers; the dealers' resale markup
IS_SALES = REVENUE * IS_SHARE
RESALE = SHIP * (1 + MARKUP)

ENTITIES = ["Suppliers", "Meridian", "Dealer network", "End customers", "Capital markets", "Regulator"]
FLOWS = [   # Table 2.6: id, from, to, what flows, type, value a year ($ million; None where the book gives none)
    ("F1", "Suppliers", "Meridian", "components", "goods", None),
    ("F2", "Meridian", "Suppliers", "payments", "capital", None),
    ("F3", "Meridian", "Dealer network", "shipments", "goods", SHIP),
    ("F4", "Dealer network", "Meridian", "payments", "capital", SHIP),
    ("F5", "Dealer network", "End customers", "sales", "goods", RESALE),
    ("F6", "End customers", "Dealer network", "payments", "capital", RESALE),
    ("F7", "Dealer network", "Meridian", "market intelligence", "information", None),
    ("F8", "Capital markets", "Meridian", "funding", "capital", None),
    ("F9", "Regulator", "Meridian", "rules", "information", None),
]
CLASSES = ("interior", "input", "output", "environmental")

def classify(src, dst, B):
    # Definition 2.3: a flow's class depends only on whether its two ends lie inside the boundary B
    return {(True, True): "interior", (False, True): "input", (True, False): "output",
            (False, False): "environmental"}[(src in B, dst in B)]

def classification(B, flows=FLOWS):
    return {f[0]: classify(f[1], f[2], B) for f in flows}

def counts(cls):
    return {c: sum(v == c for v in cls.values()) for c in CLASSES}

def goods_output(cls):
    # the goods that leave the perimeter, valued: what each team reports as "sales"
    return sum(v for fid, s, d, what, kind, v in FLOWS if kind == "goods" and cls[fid] == "output" and v is not None)

print(f"Industrial Systems' sales {IS_SALES:,.0f} ({100*IS_SHARE:.0f}% of {REVENUE:,.0f}); "
      f"shipped to dealers {SHIP:,.0f} = {100*SHIP/IS_SALES:.1f}% of it")
print(f"dealers' sales to end customers {SHIP:,.0f} x {1+MARKUP:.2f} = {RESALE:,.0f}")
print(f"{len(ENTITIES)} entities, {len(FLOWS)} flows; types used:", sorted({f[4] for f in FLOWS}))

## Panel 1 — Draw boundary A (Worked Example 2.1, Step 2; Table 2.6)

AXIOM Lab 1.2, step 1: place only Meridian inside the perimeter and record what each flow becomes. By
Definition 2.3 a flow is **interior** if both ends lie inside, an **input** if it enters, an **output** if it
leaves, and **environmental** if neither end is inside. Meridian's goods output to the market is its shipments to
dealers, \$1,540 million; the dealers' sales to end customers do not touch the boundary at all, and Meridian
learns about end demand only through the dealers' market intelligence, an input.

In [ ]:
A = {"Meridian"}
clsA = classification(A)
print("flow  from -> to                          what flows            boundary A")
for fid, s, d, what, kind, v in FLOWS:
    val = f", {v:,.0f}" if v else ""
    print(f"{fid:4s}  {s + ' -> ' + d:36s} {what + val:21s} {clsA[fid]}")
nA = counts(clsA)
print("\ninterior / input / output / environmental:", " / ".join(str(nA[c]) for c in CLASSES))
print(f"goods output under boundary A: {goods_output(clsA):,.0f}")

## Panel 2 — Move the dealers inside (Worked Example 2.1, Steps 3–5; Theorem 2.2(i)–(ii))

AXIOM Lab 1.2, step 2: drag the dealer network across the line and read the conservation panel. Shipments,
dealer payments and market intelligence become interior; the goods output is now the dealers' sales to end
customers, $1{,}540 \times 1.20 = 1{,}848$, and end-customer payments become an input. The four classes are
computed below from their definitions, separately, so the panel can test Theorem 2.2: every flow lands in
exactly one class, and the set of flows is the same under both boundaries.

In [ ]:
B = {"Meridian", "Dealer network"}
clsB = classification(B)
print("flow  boundary A        boundary B")
for fid in clsA:
    mark = "   relabeled" if clsA[fid] != clsB[fid] else ""
    print(f"{fid:4s}  {clsA[fid]:16s}  {clsB[fid]:16s}{mark}")
nB = counts(clsB)
print("\ninterior / input / output / environmental:  A", " / ".join(str(nA[c]) for c in CLASSES),
      "   B", " / ".join(str(nB[c]) for c in CLASSES))

def classes_by_definition(Bx, flows=FLOWS):
    # the four sets of Definition 2.3, each from its own membership test
    return {"interior": {f[0] for f in flows if f[1] in Bx and f[2] in Bx},
            "input": {f[0] for f in flows if f[1] not in Bx and f[2] in Bx},
            "output": {f[0] for f in flows if f[1] in Bx and f[2] not in Bx},
            "environmental": {f[0] for f in flows if f[1] not in Bx and f[2] not in Bx}}

def conservation(Bx, flows=FLOWS):
    S = classes_by_definition(Bx, flows)
    ids = {f[0] for f in flows}
    not_once = sum(sum(f in s for s in S.values()) != 1 for f in ids)      # Theorem 2.2(i)
    return not_once, set().union(*S.values()) == ids, sum(len(s) for s in S.values())   # (ii)
for name, Bx in (("A", A), ("B", B)):
    bad, same, total = conservation(Bx)
    print(f"conservation panel, boundary {name}: {total} flows; flows not in exactly one class: {bad}; "
          f"the same nine flows: {'yes' if same else 'no'}")

changed = [f for f in clsA if clsA[f] != clsB[f]]
gap = goods_output(clsB) - goods_output(clsA)
print(f"\nrelabeled by the move: {', '.join(changed)} ({len(changed)} flows); no flow created or destroyed")
print(f"goods output: A {goods_output(clsA):,.0f}, B {goods_output(clsB):,.0f}; difference {gap:,.0f} = "
      f"{100*gap/goods_output(clsA):.0f}% of shipments, the dealers' gross margin {RESALE - SHIP:,.0f}")
money = {name: {c: sum(v for f, s, d, w, k, v in FLOWS if v and cl[f] == c) for c in CLASSES} for name, cl in (("A", clsA), ("B", clsB))}
for name in ("A", "B"):
    print(f"valued flows by class, boundary {name}:", {c: round(x) for c, x in money[name].items()}, "total", round(sum(money[name].values())))
print(f"end-customer demand: sales F5 {clsA['F5']} -> {clsB['F5']}, payments F6 {clsA['F6']} -> {clsB['F6']}: "
      "seen directly, if Meridian can get the data")
print("controllability: unchanged; the dealers remain independent businesses")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7.6, 3.6), gridspec_kw={"width_ratios": [1.6, 1]})
xs = np.arange(len(CLASSES))
ax1.bar(xs - 0.2, [nA[c] for c in CLASSES], 0.4, color="#616161", label="boundary A: dealers outside")
ax1.bar(xs + 0.2, [nB[c] for c in CLASSES], 0.4, color="#1F4E79", label="boundary B: dealers inside")
for i, c in enumerate(CLASSES):
    ax1.text(i - 0.2, nA[c] + 0.1, str(nA[c]), ha="center", fontsize=8)
    ax1.text(i + 0.2, nB[c] + 0.1, str(nB[c]), ha="center", fontsize=8)
ax1.set_xticks(xs, CLASSES)
ax1.set(ylabel="flows (count)", ylim=(0, 6.5))
ax1.legend(frameon=False, fontsize=8)
ax2.bar([0, 1], [goods_output(clsA), goods_output(clsB)], 0.6, color=["#616161", "#1F4E79"])
ax2.bar([1], [gap], 0.6, bottom=[goods_output(clsA)], color="#E65100", label=f"dealers' margin {gap:,.0f}")
for i, v in enumerate((goods_output(clsA), goods_output(clsB))):
    ax2.text(i, v + 25, f"{v:,.0f}", ha="center", fontsize=8)
ax2.set_xticks([0, 1], ["A", "B"])
ax2.set(ylabel="goods output ($ million a year)", ylim=(0, 2150))
ax2.legend(frameon=False, fontsize=8, loc="upper left")
fig.suptitle("Two boundaries, nine flows: goods output 1,540 or 1,848 (seed 26102)", fontsize=10)
plt.tight_layout(); plt.show()

## Panel 3 — Repeat with other entities: which moves change nothing (Theorem 2.2(iii); Corollary 2.B.2)

AXIOM Lab 1.2, step 2, second part. Theorem 2.2(iii): two boundaries classify the flows identically exactly when
they agree on every entity that takes part in some flow. In Meridian's map all six entities take part in a flow,
so every one of the $2^6 = 64$ boundaries gives a different classification and every move relabels something.
A boundary debate is idle only for entities outside the flows being analyzed (Online Appendix Corollary 2.B.2):
for the goods question alone, capital markets and the regulator can sit on either side of the line.

In [ ]:
ALL = [frozenset(e for e, bit in zip(ENTITIES, bits) if bit) for bits in itertools.product((0, 1), repeat=len(ENTITIES))]

def theorem_iii_breaches(flows):
    # pairs of boundaries whose classifications agree but which differ on an entity that takes part in a flow, or vice versa
    EF = {f[1] for f in flows} | {f[2] for f in flows}
    sig = [tuple(classify(f[1], f[2], Bx) for f in flows) for Bx in ALL]
    return sum((sig[i] == sig[j]) != ((ALL[i] & EF) == (ALL[j] & EF)) for i in range(len(ALL)) for j in range(len(ALL))), len(set(sig)), EF

worst = max(conservation(Bx)[0] for Bx in ALL)
totals = sorted({conservation(Bx)[2] for Bx in ALL})
breaches, n_distinct, EF = theorem_iii_breaches(FLOWS)
print(f"{len(ALL)} boundaries: flows counted per boundary {totals}; flows not in exactly one class, worst case: {worst}")
print(f"distinct classifications: {n_distinct}; pairs that break Theorem 2.2(iii): {breaches}")
print("\nmove one entity, starting from boundary A        flows relabeled")
moves = {}
for e in ENTITIES:
    Bx = A ^ {e}
    rel = [f for f, c in classification(Bx).items() if c != clsA[f]]
    moves[e] = len(rel)
    verb = "outside" if e in A else "inside"
    print(f"  {e + ' ' + verb:28s}                   {len(rel)}: {', '.join(rel)}")
print("moves that change nothing:", [e for e in ENTITIES if moves[e] == 0] or "none: every entity takes part in some flow")

distinct = {}
for kind in ("goods", "capital", "information"):
    sub = [f for f in FLOWS if f[4] == kind]
    b, n, EFk = theorem_iii_breaches(sub)
    distinct[kind] = n
    free = [e for e in ENTITIES if e not in EFk]
    print(f"{kind:11s} flows {', '.join(f[0] for f in sub):14s}: {n:2d} distinct classifications; breaches {b}; free to move: {free}")

rows = []
for Bx in ALL:
    if "Meridian" in Bx:
        n = counts(classification(Bx))
        rows.append((len(Bx), -n["interior"], Bx == A, Bx == B, [n[c] for c in CLASSES]))
rows.sort(key=lambda r: (r[0], r[1]))
fig, ax = plt.subplots(figsize=(7.6, 3.6))
bottom = np.zeros(len(rows))
for j, (c, col) in enumerate(zip(CLASSES, ("#1F4E79", "#E65100", "#B71C1C", "#999999"))):
    v = np.array([r[4][j] for r in rows])
    ax.bar(np.arange(len(rows)), v, 0.8, bottom=bottom, color=col, label=c)
    bottom += v
for i, r in enumerate(rows):
    if r[2] or r[3]:
        ax.text(i, 9.15, "A" if r[2] else "B", ha="center", fontsize=8, fontweight="bold")
ax.set(xlabel="the 32 boundaries that keep Meridian inside, by number of entities inside", ylabel="flows (count)",
       ylim=(0, 10), title="Every boundary classifies the same nine flows (seed 26102)")
ax.legend(frameon=False, fontsize=8, ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.2))
plt.tight_layout(); plt.show()

**The rule for the board (Worked Example 2.1, Step 6).** The \$308 million gap is a boundary difference, not a
performance difference: the two teams agree on the facts. Options must be compared under one declared boundary,
and every result reported as conditional on it. Boundary B suits the digital option's question (what do end
customers buy, and could Meridian serve them directly?); boundary A suits the turnaround's (what does Meridian's
own operation earn?). Drawing the dealers inside changes the labels, not control: they remain independent.

## Panel 4 — Change levels (Definition 2.4; Figure 2.3)

AXIOM Lab 1.2, step 3: zoom from group to division to unit. Meridian is a system of systems: the group, its
three divisions and their units (Figure 2.3). By Definition 2.4 a subsystem's environment is its parent's
environment **plus its sibling subsystems**, so the environment changes with the level. The same boundary
machinery then classifies the coupling Figure 2.3 draws between the divisions — Digital Solutions' demands for
engineers and capital reach Industrial Systems through it.

In [ ]:
HIERARCHY = {"Meridian Group": ["Industrial Systems", "Digital Solutions", "Advanced Materials"],
             "Industrial Systems": ["Plants", "Service"], "Digital Solutions": ["Platform", "Apps"],
             "Advanced Materials": ["Polymers", "Composites"]}
PARENT = {c: p for p, cs in HIERARCHY.items() for c in cs}
DIVISIONS = HIERARCHY["Meridian Group"]
UNITS = [u for d in DIVISIONS for u in HIERARCHY[d]]

def environment(s):
    # Definition 2.4: the parent's environment plus the sibling subsystems
    if s not in PARENT:
        return ["the external environment"]
    p = PARENT[s]
    return environment(p) + [x for x in HIERARCHY[p] if x != s]

print(f"Meridian: {len(DIVISIONS)} divisions, {len(UNITS)} units (Figure 2.3)\n")
print("level     system              inside                 environment")
for level, s in (("group", "Meridian Group"), ("division", "Digital Solutions"), ("unit", "Platform")):
    inside = f"{len(DIVISIONS)} divisions, {len(UNITS)} units" if s == "Meridian Group" else ", ".join(HIERARCHY.get(s, ["-"]))
    env = environment(s)
    print(f"{level:9s} {s:19s} {inside:22s} {env[0]}" + "".join(f" + {x}" for x in env[1:]))
siblings_DS = [x for x in environment("Digital Solutions") if x in DIVISIONS]
print(f"Digital Solutions' environment holds {len(siblings_DS)} sibling divisions and their {2 * len(siblings_DS)} units")

SHORT = {"Industrial Systems": "IS", "Digital Solutions": "DS", "Advanced Materials": "AM"}
COUPLING = [("C1", "Industrial Systems", "Digital Solutions"), ("C2", "Digital Solutions", "Industrial Systems"),
            ("C3", "Digital Solutions", "Advanced Materials"), ("C4", "Advanced Materials", "Digital Solutions")]
print("\nthe coupling Figure 2.3 draws, both ways:  " + "".join(f"{c[0] + ' ' + SHORT[c[1]] + '->' + SHORT[c[2]]:15s}" for c in COUPLING))
levels = {}
for name, Bx in (("the group", set(DIVISIONS)), ("Industrial Systems", {"Industrial Systems"}),
                 ("Digital Solutions", {"Digital Solutions"}), ("Advanced Materials", {"Advanced Materials"})):
    cl = {c[0]: classify(c[1], c[2], Bx) for c in COUPLING}
    levels[name] = cl
    print(f"boundary around {name:20s}       " + "".join(f"{cl[c[0]]:15s}" for c in COUPLING))
print("interior at the group level; at the division level it crosses the boundary of each division it joins,"
      "\nand is environmental for the third")

## Panel 5 — Close the loop: feedback (Definition 2.5; Proposition 2.2; Tables 1.9 and 5.5)

AXIOM Lab 1.2, step 4: the feedback bench runs an efficiency example. Here it is Meridian's: operational
efficiency is 71 today (Table 3.1), and the operational turnaround of Chapter 1's model takes it toward a target
of 79 at a speed of 0.60 a year (Online Appendix Table 1.B.1). Read as a rule, that program is **state
feedback** (Definition 2.5): each year it acts on the gap, $u_k = K\,(79 - x_k)$, with
$K = 1 - e^{-0.60} = 0.451$, so $x_{k+1} = x_k + u_k + w_k$ and the gap shrinks by the factor 0.549 a year
(Worked Example 5.1). Switch the feedback off and the gap stays where it is; vary the gain and the loop speeds
up, then overshoots. By Proposition 2.2 the company with its rule is again a system: one path for every
environment.

In [ ]:
X0_EFF, TARGET, SPEED = 71.0, 79.0, 0.60      # efficiency today; the turnaround's target and speed (Chapter 1 model)
K_T = 1 - np.exp(-SPEED)                       # share of the gap the program closes each year

def run(gain, years=20, x0=X0_EFF, w=None):
    x = [x0]
    for k in range(years):
        u = gain * (TARGET - x[-1])            # state feedback: the decision is set by the rule from the state
        x.append(x[-1] + u + (0.0 if w is None else w[k]))
    return np.array(x)

path = run(K_T)
print(f"gain of the turnaround's rule K = {K_T:.4f}; the gap is multiplied by {1 - K_T:.4f} a year")
print("year            " + "".join(f"{k:8d}" for k in (0, 1, 3, 5, 11)))
print("feedback on     " + "".join(f"{path[k]:8.2f}" for k in (0, 1, 3, 5, 11)), "  (Table 5.5: 71.0, 74.6, 77.7, 78.6, 79.0)")
print("feedback off    " + "".join(f"{run(0.0)[k]:8.2f}" for k in (0, 1, 3, 5, 11)), "  (no rule: the gap of 8 points stays)")
gains = [0.0, 0.25, K_T, 1.0, 1.6, 2.2]
for g in gains:
    p = run(g)
    years_in = next((k for k in range(len(p)) if np.all(np.abs(p[k:] - TARGET) <= 0.5)), None)
    shape = "never" if years_in is None else f"from year {years_in}"
    print(f"gain {g:5.3f}: years 1-4 {np.round(p[1:5], 2)}; within half a point of 79 {shape}")

def closed_loop(x, w, K=K_T):
    # Proposition 2.2: substitute the rule into the transition, f~(x, w) = f(x, phi(x), w): one map of state and input
    return (1 - K) * x + K * TARGET + w
rng = np.random.default_rng(SEED)
W = rng.normal(0.0, 1.0, size=(1000, 20))     # 1,000 environment paths (any paths would do)
gap22 = 0.0
for w in W:
    p_map = [X0_EFF]
    for wk in w:
        p_map.append(closed_loop(p_map[-1], wk))
    gap22 = max(gap22, float(np.abs(run(K_T, w=w) - np.array(p_map)).max()))
print(f"Proposition 2.2: rule-then-transition against the closed-loop map on 1,000 environment paths (seed {SEED}): "
      f"largest difference {gap22:.1e}")

fig, ax = plt.subplots(figsize=(7.6, 3.6))
yrs = np.arange(21)
for g, col, ls, lab in ((0.0, "#999999", "-", "feedback off"), (0.25, "#616161", "--", "gain 0.25"),
                        (K_T, "#1F4E79", "-", "the turnaround's rule, gain 0.451"), (1.0, "#E65100", "--", "gain 1.0"),
                        (1.6, "#B71C1C", ":", "gain 1.6: overshoots")):
    ax.plot(yrs, run(g), ls, color=col, lw=2 if g == K_T else 1.4, label=lab)
ax.plot([1, 3, 5, 11], [74.6, 77.7, 78.6, 79.0], "o", color="#1F4E79", ms=5, label="Table 5.5")
ax.set_xticks(range(0, 21, 2))
ax.set(xlabel="years", ylabel="operational efficiency (index, 0-100)", ylim=(70, 85),
       title="Feedback holds efficiency at the target; the gain sets the pace (seed 26102)")
ax.legend(frameon=False, fontsize=8, ncol=2); plt.tight_layout(); plt.show()

**Output feedback needs a reporting lag (Online Appendix Remark 2.B.1).** Proposition 2.2 uses the fact that
outputs depend on the state alone. If a reported figure responds instantly to the decision, $y = g(x, u) = u$,
output feedback must solve $u = \varphi(g(x, u))$: with $\varphi(y) = y + 1$ there is no solution, with
$\varphi(y) = y^2$ there are two, $u = 0$ and $u = 1$. A reporting lag, $u_k = \varphi(y_{k-1})$, removes the
problem: every rule then gives one path.

In [ ]:
from numpy.polynomial import polynomial as Pn
def loop_solutions(phi_coeffs):
    # real solutions of u = phi(u) for a polynomial phi (coefficients from the constant term up)
    c = np.array(phi_coeffs, float); c = np.pad(c, (0, max(0, 2 - len(c)))); c[1] -= 1.0
    c = np.trim_zeros(c, "b")
    if len(c) <= 1:
        return []                                   # a nonzero constant: no solution
    r = Pn.polyroots(c)
    return sorted(float(x.real) for x in r if abs(x.imag) < 1e-12)
sol_plus = loop_solutions([1.0, 1.0])          # phi(y) = y + 1
sol_square = loop_solutions([0.0, 0.0, 1.0])   # phi(y) = y^2
print("phi(y) = y + 1:  solutions of u = phi(u):", sol_plus or "none")
print("phi(y) = y^2:    solutions of u = phi(u):", sol_square)
u = [0.0]
for k in range(5):
    u.append(u[-1] + 1.0)                        # with a one-period lag: u_k = phi(y_{k-1}), y = u
print("with a one-period reporting lag, phi(y) = y + 1 gives one path:", u)

## Panel 6 — Close the loop: adaptation, and a shift in the environment (Definition 2.6)

AXIOM Lab 1.2, step 4, second part. After year 5, when the turnaround has delivered 78.6 (Table 1.9), switch on
an adaptation law of the form of Online Appendix 2.B — the rule's gain $\theta$ is revised upward when the
squared gap is large and downward when it is small, $\theta_{k+1} = \theta_k + \eta\,\bigl((x_k - 79)^2 - c\bigr)$ —
and let the environment start taking efficiency away each year. The book prints no numbers for this step: the
shift, the speed $\eta$ and the tolerance $c$ below are bench settings to vary, not Meridian facts. The fixed
rule settles a predictable $s/K$ points short of 79; the adaptive rule holds closer, but how it gets there, and
whether it does, depends on the shift and on how fast it adapts.

In [ ]:
SHIFT_YEAR, ETA, C_TOL = 5, 0.05, 1.0         # bench settings: when the environment shifts; adaptation speed; tolerated squared gap

def bench(shift, adapt=True, eta=ETA, c=C_TOL, years=20, theta0=K_T, x0=X0_EFF):
    # the fixed rule until SHIFT_YEAR; then the environment takes `shift` points a year and, if adapt, the law is on
    x, th = [x0], [theta0]
    for k in range(years):
        w = -shift if k >= SHIFT_YEAR else 0.0
        gap = x[-1] - TARGET
        x.append(x[-1] + th[-1] * (TARGET - x[-1]) + w)
        th.append(th[-1] + (eta * (gap ** 2 - c) if adapt and k >= SHIFT_YEAR else 0.0))
        if not 0.0 <= x[-1] <= 100.0:            # the index leaves its range: the loop has lost control
            break
    return np.array(x), np.array(th)

print("shift a year   fixed rule settles at (79 - s/K)    adaptive rule: year 20, gain")
res = {}
for s, eta in ((1.0, ETA), (2.0, ETA), (3.0, ETA), (2.0, 2 * ETA)):
    xf, _ = bench(s, adapt=False, years=40)
    xa, th = bench(s, eta=eta)
    res[(s, eta)] = (xf, xa, th)
    tail = (f"{xa[20]:6.2f}, gain {th[20]:4.2f}" if len(xa) > 20 else f"leaves the 0-100 range in year {len(xa) - 1}")
    note = "" if eta == ETA else f"   (adapting twice as fast, eta {eta:.2f})"
    print(f"   {s:4.1f}            {xf[-1]:6.2f}   (short {s / K_T:4.2f})                 {tail}{note}")
xf1, xa1, th1 = res[(1.0, ETA)]
print("\na model that freezes the rule at 0.451 (and knows the shift) forecasts the adaptive company with errors of")
print("   " + ", ".join(f"year {k}: {xa1[k] - bench(1.0, adapt=False)[0][k]:+.2f}" for k in (6, 7, 8, 10, 15, 20)))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7.6, 3.6), gridspec_kw={"width_ratios": [1.5, 1]})
yrs = np.arange(21)
off = np.concatenate([path[:SHIFT_YEAR + 1], path[SHIFT_YEAR] - np.arange(1, 16) * 1.0])
ax1.plot(yrs, off, color="#999999", lw=1.2, label="feedback off from year 5 (shift 1)")
for s, ls in ((1.0, "-"), (2.0, "--")):
    xf, xa, th = res[(s, ETA)]
    ax1.plot(yrs, xf[:21], ls, color="#1F4E79", lw=1.8, label=f"fixed rule, shift {s:.0f} a year")
    ax1.plot(yrs, xa[:21], ls, color="#E65100", lw=1.8, label=f"adaptive rule, shift {s:.0f} a year")
    ax2.plot(yrs, th[:21], ls, color="#E65100", lw=1.8)
ax2.axhline(K_T, color="#1F4E79", lw=1.8)
ax1.axhline(TARGET, color="#999999", lw=0.6); ax1.axvline(SHIFT_YEAR, color="#999999", lw=0.6, ls=":")
for ax in (ax1, ax2):
    ax.set_xticks(range(0, 21, 5))
ax1.set(xlabel="years", ylabel="operational efficiency (index)", ylim=(70, 80))
ax2.set(xlabel="years", ylabel="gain (share of the gap a year)")
fig.legend(*ax1.get_legend_handles_labels(), loc="lower center", ncol=3, frameon=False, fontsize=7)
fig.suptitle("After the shift: the adaptive rule holds closer to 79, its gain swings (seed 26102)", fontsize=10)
plt.tight_layout(rect=(0, 0.12, 1, 1)); plt.show()

## Panel 7 — The adaptive company is an ordinary system (Proposition 2.3; Theorem 2.1; Online Appendix 2.B)

AXIOM Lab 1.2, step 4, continued: why the adaptive rule can still be analyzed. Count the rule as part of the
state, $z_k = (x_k, \theta_k)$, and the two-loop company is one map $z_{k+1} = F(z_k, w_k)$, to which Theorem 2.1
applies. Online Appendix 2.B writes this out for the scalar company $x_{k+1} = a x_k + u_k + w_k$,
$u_k = -\theta_k x_k$, $\theta_{k+1} = \theta_k + \eta(x_k^2 - c)$, whose map is
$F(z, w) = ((a - \theta) x + w,\ \theta + \eta(x^2 - c))$. Theorem 2.1(ii) says two histories that reach the same
state have the same future; the Online Appendix adds what a fixed-rule model misses: the same $x$ with different
rules $\theta$ has different futures. Its $a$, $\eta$ and $c$ are symbolic; the runs below use bench values
($a = 0.9$, $\eta = 0.05$, $c = 1$).

In [ ]:
A_OWN = 0.9                                      # bench value of the deviation's own persistence a
def oa_two_loops(x0, th0, w, eta, c, a=A_OWN):   # the Online Appendix's company, as written: rule, decision, two laws
    x, th = [x0], [th0]
    for wk in w:
        u = -th[-1] * x[-1]
        x_next = a * x[-1] + u + wk
        th.append(th[-1] + eta * (x[-1] ** 2 - c)); x.append(x_next)
    return np.array(x), np.array(th)

def oa_F(z, wk, eta, c, a=A_OWN):                # the same company as one map of the augmented state
    x, th = z
    return np.array([(a - th) * x + wk, th + eta * (x ** 2 - c)])

rng = np.random.default_rng(SEED)
gap_F = 0.0
for _ in range(1000):                            # 1,000 random histories: starts, rules, environments
    x0, th0 = rng.uniform(-2, 2), rng.uniform(0.1, 0.6)
    w = rng.normal(0.0, 0.5, 12)
    xs, ths = oa_two_loops(x0, th0, w, eta=0.05, c=1.0)
    z = np.array([x0, th0]); zs = [z]
    for wk in w:
        z = oa_F(z, wk, 0.05, 1.0); zs.append(z)
    gap_F = max(gap_F, float(np.abs(np.array(zs) - np.column_stack([xs, ths])).max()))
print(f"Proposition 2.3: the map F reproduces the two-loop company on 1,000 histories (seed {SEED}); largest difference {gap_F:.1e}")

def efficiency_F(z, wk, adapt=True, eta=ETA, c=C_TOL):   # the bench's efficiency company as one map of (x, theta)
    x, th = z
    return np.array([x + th * (TARGET - x) + wk, th + (eta * ((x - TARGET) ** 2 - c) if adapt else 0.0)])

def future(z5, shift, adapt, years=15):
    z, out = np.array(z5, float), [z5[0]]
    for k in range(years):
        z = efficiency_F(z, -shift, adapt); out.append(z[0])
    return np.array(out)

x5 = path[SHIFT_YEAR]                            # 78.60, reached from 71 by the turnaround
h2 = run(K_T, years=4, x0=75.0)                  # a second history: another start ...
w4 = x5 - (h2[-1] + K_T * (TARGET - h2[-1]))     # ... and another environment, reaching the same efficiency in year 5
x5_second = h2[-1] + K_T * (TARGET - h2[-1]) + w4
fixed_1, fixed_2 = future((x5, K_T), 1.0, False), future((x5_second, K_T), 1.0, False)
same_state_gap = float(np.abs(fixed_1 - fixed_2).max())
print(f"fixed rule: two histories (from 71, and from 75 with another environment) reach {x5:.4f} and {x5_second:.4f} "
      f"in year 5;\n   facing the same shift, their futures differ by at most {same_state_gap:.1e}")
adapt_1, adapt_2 = future((x5, K_T), 1.0, True), future((x5, 2 * K_T), 1.0, True)
rule_gap = float(np.abs(adapt_1 - adapt_2).max())
print(f"adaptive rule: the same efficiency {x5:.2f} in year 5 with rules {K_T:.3f} and {2 * K_T:.3f}; their futures differ by up to "
      f"{rule_gap:.2f} points:\n   efficiency alone is no longer a sufficient statistic; (efficiency, rule) is")

## Panel 8 — Map a loop (Tables 2.4 and 2.5)

AXIOM Lab 1.2, step 5. Industrial Systems' migration loop, in the book's words: customer migration **cuts**
volume, which **raises** unit cost, which **speeds** migration (Table 2.5). Label each link by whether it moves
its target the same way (+) or the opposite way (−); a loop's polarity is the product of its links' signs, and a
push that comes back around the loop with the same sign marks a reinforcing loop. The turnaround's efficiency
program of Panel 5 is the balancing kind (Table 2.4: it opposes deviation from a target).

In [ ]:
LOOPS = {
    "Industrial Systems' migration loop": [("customer migration", "volume", -1, "migration cuts volume"),
                                            ("volume", "unit cost", -1, "less volume raises unit cost"),
                                            ("unit cost", "customer migration", +1, "higher unit cost speeds migration")],
    "the turnaround's efficiency program": [("efficiency gap", "program effort", +1, "a wider gap draws more effort"),
                                             ("program effort", "efficiency", +1, "effort raises efficiency"),
                                             ("efficiency", "efficiency gap", -1, "higher efficiency narrows the gap")],
}
polarity = {}
for name, links in LOOPS.items():
    sign = int(np.prod([l[2] for l in links]))
    polarity[name] = sign
    print(f"{name}: " + "; ".join(f"{a} -> {b} ({'+' if s > 0 else '-'})" for a, b, s, _ in links))
    print(f"   opposite links {sum(l[2] < 0 for l in links)}, polarity {'+' if sign > 0 else '-'}: "
          f"{'reinforcing' if sign > 0 else 'balancing'}; a push of +1 comes back as {sign:+d}")

## Validation — the book's numbers, and agreement with `DCT_V1_Ch02_Lab.xlsx`

Each checkpoint is a number printed in Chapter 2 (Worked Example 2.1, Table 2.6, Figure 2.3), in Online
Appendix 2.B, or, for the feedback bench, in Tables 1.9 and 5.5 and Worked Example 5.1. Theorem checks compare
with what the printed theorem asserts. The workbook recomputes the ones marked *workbook* with Excel formulas and
shows PASS against the same engine values on its `Checks` sheet.

In [ ]:
TABLE_2_6 = {"F1": ("input", "input"), "F2": ("output", "output"), "F3": ("output", "interior"),
             "F4": ("input", "interior"), "F5": ("environmental", "output"), "F6": ("environmental", "input"),
             "F7": ("input", "interior"), "F8": ("input", "input"), "F9": ("input", "input")}
matchA = sum(clsA[f] == TABLE_2_6[f][0] for f in TABLE_2_6)
matchB = sum(clsB[f] == TABLE_2_6[f][1] for f in TABLE_2_6)
checks = [
    ("Industrial Systems' sales, $2.56bn (workbook)", IS_SALES, 2560, 5),
    ("share shipped to dealers, about 60% (workbook)", 100 * SHIP / IS_SALES, 60, 0.5),
    ("dealers' sales to end customers (workbook)", RESALE, 1848, 0.5),
    ("goods output, boundary A (workbook)", goods_output(clsA), 1540, 0.5),
    ("goods output, boundary B (workbook)", goods_output(clsB), 1848, 0.5),
    ("the gap: dealers' gross margin (workbook)", gap, 308, 0.5),
    ("the gap as a share of shipments, % (workbook)", 100 * gap / goods_output(clsA), 20, 0.05),
    ("entities in the map", len(ENTITIES), 6, 0),
    ("flows in the map", len(FLOWS), 9, 0),
    ("boundary A: interior flows (workbook)", nA["interior"], 0, 0),
    ("boundary A: inputs (workbook)", nA["input"], 5, 0),
    ("boundary A: outputs (workbook)", nA["output"], 2, 0),
    ("boundary A: environmental flows (workbook)", nA["environmental"], 2, 0),
    ("boundary B: interior flows (workbook)", nB["interior"], 3, 0),
    ("boundary B: inputs (workbook)", nB["input"], 4, 0),
    ("boundary B: outputs (workbook)", nB["output"], 2, 0),
    ("boundary B: environmental flows (workbook)", nB["environmental"], 0, 0),
    ("Table 2.6 rows reproduced, boundary A (workbook)", matchA, 9, 0),
    ("Table 2.6 rows reproduced, boundary B (workbook)", matchB, 9, 0),
    ("flows relabeled by the move, Table 2.6 (workbook)", len(changed), 5, 0),
    ("Theorem 2.2(i): flows not in exactly one class (workbook)", worst, 0, 0),
    ("Theorem 2.2(ii): flows under every boundary (workbook)", min(totals), 9, 0),
    ("Theorem 2.2(iii): breaching pairs of boundaries", breaches, 0, 0),
    ("Figure 2.3: Meridian's divisions (workbook)", len(DIVISIONS), 3, 0),
    ("Figure 2.3: units (workbook)", len(UNITS), 6, 0),
    ("Definition 2.4: Digital Solutions' sibling divisions (workbook)", len(siblings_DS), 2, 0),
    ("turnaround efficiency, year 1, Table 5.5 (workbook)", path[1], 74.6, 0.05),
    ("turnaround efficiency, year 3, Table 5.5 (workbook)", path[3], 77.7, 0.05),
    ("turnaround efficiency, year 5, Tables 1.9, 5.5 (workbook)", path[5], 78.6, 0.05),
    ("turnaround efficiency, year 11, Table 5.5 (workbook)", path[11], 79.0, 0.05),
    ("gap factor a year, Worked Example 5.1 (workbook)", 1 - K_T, 0.549, 5e-4),
    ("Proposition 2.2: closed-loop map vs rule and transition (workbook)", gap22, 0, 1e-9),
    ("Remark 2.B.1: solutions of u = u + 1 (workbook)", len(sol_plus), 0, 0),
    ("Remark 2.B.1: solutions of u = u^2 (workbook)", len(sol_square), 2, 0),
    ("Remark 2.B.1: the larger solution (workbook)", max(sol_square), 1, 1e-12),
    ("Proposition 2.3: augmented map vs two loops (workbook)", gap_F, 0, 1e-9),
    ("Theorem 2.1(ii): same state, futures differ by", same_state_gap, 0, 1e-9),
    ("migration loop: polarity +1, reinforcing (workbook)", polarity["Industrial Systems' migration loop"], 1, 0),
    ("efficiency program: polarity -1, balancing (workbook)", polarity["the turnaround's efficiency program"], -1, 0),
]
ok = 0
for name, got, want, tol in checks:
    status = "PASS" if abs(got - want) <= tol else "FAIL"; ok += status == "PASS"
    print(f"{name:64s} {got:10.4f}   book {want:9.4f}   {status}")
print(f"\n{ok} of {len(checks)} checkpoints PASS")

**Next.** Chapter 3 gives the state its numbers: readings with units, distances on a declared scale, and
transformations as operators that work only from certain states. AXIOM-02 runs the same world interactively
(Lab 1.2); this notebook and the workbook are its reproducible record.